In [1]:
import pandas as pd

df = pd.read_csv("annual-co2-emissions-per-country.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())
print("\nMissing values:")
print(df.isnull().sum())

print("\nYear range:", df["Year"].min(), "to", df["Year"].max())
print("\nUnique countries:", df["Entity"].nunique())
print("\nTop 10 CO2 emitters ever:")
print(df.groupby("Entity")["Annual CO₂ emissions"].sum().sort_values(ascending=False).head(10))

Shape: (29384, 4)

Columns: ['Entity', 'Code', 'Year', 'Annual CO₂ emissions']

First 5 rows:
        Entity Code  Year  Annual CO₂ emissions
0  Afghanistan  AFG  1949               14656.0
1  Afghanistan  AFG  1950               84272.0
2  Afghanistan  AFG  1951               91600.0
3  Afghanistan  AFG  1952               91600.0
4  Afghanistan  AFG  1953              106256.0

Missing values:
Entity                     0
Code                    3311
Year                       0
Annual CO₂ emissions       0
dtype: int64

Year range: 1750 to 2024

Unique countries: 247

Top 10 CO2 emitters ever:
Entity
World                            1.849124e+12
High-income countries            1.151674e+12
OECD (GCP)                       9.891896e+11
Non-OECD (GCP)                   8.094838e+11
Asia                             6.256351e+11
Europe                           5.528708e+11
Europe (GCP)                     5.484695e+11
Asia (GCP)                       5.446609e+11
Upper-middle-income c

In [2]:
import os
import pandas as pd
os.makedirs("output", exist_ok=True)

# Load raw data
df = pd.read_csv("annual-co2-emissions-per-country.csv")

# Step 1 — keep only actual countries (remove regions/groups)
# Countries have a 3-letter ISO code. Regions and groups either have no code
# or an "OWID_" code (e.g. OWID_WRL = World), so a notna() check is not enough:
# it would keep World, continents and income groups and double-count emissions.
df_countries = df[df["Code"].str.len() == 3]

# Step 2 — rename columns to be cleaner
df_countries = df_countries.rename(columns={
    "Entity": "country",
    "Code": "country_code",
    "Year": "year",
    "Annual CO₂ emissions": "co2_emissions"
})

# Step 3 — filter from 1900 onwards (more relevant for our story)
df_countries = df_countries[df_countries["year"] >= 1900]

# Step 4 — remove any remaining nulls
df_countries = df_countries.dropna()

# Step 5 — check our clean data
print("Clean shape:", df_countries.shape)
print("\nYear range:", df_countries["year"].min(), "to", df_countries["year"].max())
print("\nSample:")
print(df_countries.head())

# Step 6 — save to output folder
df_countries.to_csv("output/co2_clean.csv", index=False)
print("\nSaved to output/co2_clean.csv")

Clean shape: (19787, 4)

Year range: 1900 to 2024

Sample:
       country country_code  year  co2_emissions
0  Afghanistan          AFG  1949        14656.0
1  Afghanistan          AFG  1950        84272.0
2  Afghanistan          AFG  1951        91600.0
3  Afghanistan          AFG  1952        91600.0
4  Afghanistan          AFG  1953       106256.0

Saved to output/co2_clean.csv


In [3]:
import pandas as pd

# Load clean data
df = pd.read_csv("output/co2_clean.csv")

# --- Insight 1: Top 10 countries by total CO2 (actual countries only) ---
print("=== Top 10 biggest CO2 emitters (1900-2024) ===")
top10 = df.groupby("country")["co2_emissions"].sum().sort_values(ascending=False).head(10)
print(top10.to_string())

# --- Insight 2: Global emissions by decade ---
print("\n=== Global CO2 by decade ===")
df["decade"] = (df["year"] // 10) * 10
by_decade = df.groupby("decade")["co2_emissions"].sum().reset_index()
by_decade["co2_billions"] = (by_decade["co2_emissions"] / 1e9).round(2)
print(by_decade[["decade", "co2_billions"]].to_string())

# --- Insight 3: Fastest growing emitters since 2000 ---
print("\n=== Fastest growing emitters (2000 vs 2020) ===")
y2000 = df[df["year"] == 2000].set_index("country")["co2_emissions"]
y2020 = df[df["year"] == 2020].set_index("country")["co2_emissions"]
growth = ((y2020 - y2000) / y2000 * 100).dropna().sort_values(ascending=False).head(10)
print(growth.round(1).to_string())

=== Top 10 biggest CO2 emitters (1900-2024) ===
country
United States     4.248530e+11
China             2.850870e+11
Russia            1.223769e+11
Germany           8.907158e+10
Japan             6.945169e+10
India             6.596946e+10
United Kingdom    6.374904e+10
France            3.646638e+10
Canada            3.538341e+10
Ukraine           3.109969e+10

=== Global CO2 by decade ===
    decade  co2_billions
0     1900         24.21
1     1910         32.78
2     1920         37.06
3     1930         38.95
4     1940         49.82
5     1950         71.24
6     1960        108.50
7     1970        167.82
8     1980        196.85
9     1990        229.15
10    2000        279.78
11    2010        342.51
12    2020        181.24

=== Fastest growing emitters (2000 vs 2020) ===
country
Laos             1804.8
Afghanistan       961.8
Cambodia          951.2
Vietnam           572.3
Tanzania          520.3
Guinea-Bissau     507.1
Burkina Faso      503.6
Mali              493.7
Mongo